# Fine Balanced Accuracy Threshold Tuning

The public score improved with probability multipliers `fit=1.15` and `unhealthy=1.05`. This notebook searches a finer grid around that region, then creates a small set of nearby submissions. Rules are tuned on validation only, then applied to test after retraining on train+validation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"

## Load Data

In [2]:
train_df = pd.read_csv("data/train_split_features_numeric.csv")
val_df = pd.read_csv("data/val_split_features_numeric.csv")
test_df = pd.read_csv("data/test_features_numeric.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

feature_cols = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]
X_train = train_df[feature_cols].astype("float32")
X_val = val_df[feature_cols].astype("float32")
X_test = test_df[feature_cols].astype("float32")
y_train_raw = train_df[TARGET_COL]
y_val_raw = val_df[TARGET_COL]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
class_names = label_encoder.classes_.tolist()
class_id = {name: int(label_encoder.transform([name])[0]) for name in class_names}

at_idx = class_id["at-risk"]
fit_idx = class_id["fit"]
unhealthy_idx = class_id["unhealthy"]

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
print("classes:", class_id)

train: (552070, 73) val: (138018, 73) test: (295753, 72)
classes: {'at-risk': 0, 'fit': 1, 'unhealthy': 2}


## Train Validation Model

In [3]:
model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.06,
    max_iter=500,
    max_leaf_nodes=31,
    l2_regularization=0.01,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbose=0,
)
model.fit(X_train, y_train)
val_proba = model.predict_proba(X_val)
print("selected iterations:", int(model.n_iter_))

selected iterations: 123


## Fine Grid Around Current Best

In [4]:
def score_multipliers(fit_mult, unhealthy_mult):
    multipliers = np.ones(len(class_names), dtype="float32")
    multipliers[fit_idx] = fit_mult
    multipliers[unhealthy_idx] = unhealthy_mult
    pred = (val_proba * multipliers).argmax(axis=1)
    recalls = recall_score(y_val, pred, average=None, labels=np.arange(len(class_names)))
    pred_labels = label_encoder.inverse_transform(pred)
    dist = pd.Series(pred_labels).value_counts(normalize=True).mul(100).to_dict()
    return {
        "fit_mult": round(float(fit_mult), 3),
        "unhealthy_mult": round(float(unhealthy_mult), 3),
        "balanced_accuracy": balanced_accuracy_score(y_val, pred),
        "accuracy": accuracy_score(y_val, pred),
        "recall_at_risk": recalls[at_idx],
        "recall_fit": recalls[fit_idx],
        "recall_unhealthy": recalls[unhealthy_idx],
        "pred_at_risk_pct": dist.get("at-risk", 0),
        "pred_fit_pct": dist.get("fit", 0),
        "pred_unhealthy_pct": dist.get("unhealthy", 0),
    }

rows = []
for fit_mult in np.round(np.arange(1.08, 1.221, 0.01), 2):
    for unhealthy_mult in np.round(np.arange(0.98, 1.121, 0.01), 2):
        rows.append(score_multipliers(fit_mult, unhealthy_mult))

fine = pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
fine.head(30)

,fit_mult,unhealthy_mult,balanced_accuracy,accuracy,recall_at_risk,recall_fit,recall_unhealthy,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,1.17,1.03,0.910935,0.872495,0.862081,0.933174,0.937549,74.870669,11.268820,13.860511
1,1.17,1.04,0.910932,0.871951,0.861381,0.933174,0.938242,74.804011,11.268820,13.927169
2,1.13,1.03,0.910921,0.874270,0.864301,0.930913,0.937549,75.074990,11.064499,13.860511
3,1.13,1.04,0.910918,0.873727,0.863600,0.930913,0.938242,75.008332,11.064499,13.927169
4,1.17,0.98,0.910894,0.875342,0.865769,0.933174,0.933738,75.220623,11.268820,13.510557
5,1.14,1.03,0.910881,0.873864,0.863803,0.931290,0.937549,75.030069,11.109421,13.860511
6,1.13,0.98,0.910880,0.877117,0.867988,0.930913,0.933738,75.424945,11.064499,13.510557
7,1.14,1.04,0.910878,0.873321,0.863102,0.931290,0.938242,74.963411,11.109421,13.927169
8,1.18,1.03,0.910864,0.872111,0.861617,0.933425,0.937549,74.829370,11.310119,13.860511
9,1.18,1.04,0.910861,0.871567,0.860917,0.933425,0.938242,74.762712,11.310119,13.927169


## Select Nearby Candidate Submissions

In [5]:
best = fine.iloc[0]

# Public score improved at 1.15/1.05, so include a few variants close to that point.
selected_points = [
    ("fine_best", float(best.fit_mult), float(best.unhealthy_mult)),
    ("public_anchor", 1.15, 1.05),
    ("fit_one_step_up", 1.16, 1.05),
    ("fit_one_step_down", 1.14, 1.05),
    ("unhealthy_one_step_up", 1.15, 1.06),
    ("unhealthy_one_step_down", 1.15, 1.04),
    ("both_slight_up", 1.16, 1.06),
    ("more_fit_less_unhealthy", 1.17, 1.03),
    ("less_fit_more_unhealthy", 1.13, 1.07),
]

selected = pd.DataFrame(selected_points, columns=["variant", "fit_mult", "unhealthy_mult"]).drop_duplicates(subset=["fit_mult", "unhealthy_mult"])
selected = selected.merge(fine, on=["fit_mult", "unhealthy_mult"], how="left")
selected.sort_values("balanced_accuracy", ascending=False)

,variant,fit_mult,unhealthy_mult,balanced_accuracy,accuracy,recall_at_risk,recall_fit,recall_unhealthy,pred_at_risk_pct,pred_fit_pct,pred_unhealthy_pct
0,fine_best,1.17,1.03,0.910935,0.872495,0.862081,0.933174,0.937549,74.870669,11.268820,13.860511
5,unhealthy_one_step_down,1.15,1.04,0.910829,0.872893,0.862579,0.931667,0.938242,74.915591,11.157240,13.927169
3,fit_one_step_down,1.14,1.05,0.910781,0.872669,0.862292,0.931290,0.938761,74.889507,11.109421,14.001072
1,public_anchor,1.15,1.05,0.910733,0.872241,0.861769,0.931667,0.938761,74.841687,11.157240,14.001072
4,unhealthy_one_step_up,1.15,1.06,0.910704,0.871698,0.861077,0.931667,0.939368,74.777203,11.157240,14.065557
7,less_fit_more_unhealthy,1.13,1.07,0.910697,0.871814,0.861204,0.930913,0.939974,74.788071,11.064499,14.147430
2,fit_one_step_up,1.16,1.05,0.910695,0.871741,0.861153,0.932169,0.938761,74.785897,11.213030,14.001072
6,both_slight_up,1.16,1.06,0.910666,0.871198,0.860461,0.932169,0.939368,74.721413,11.213030,14.065557


## Retrain On Train + Validation And Save Test Submissions

In [6]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = full_df[feature_cols].astype("float32")
y_full = label_encoder.transform(full_df[TARGET_COL])

final_model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.06,
    max_iter=int(model.n_iter_),
    max_leaf_nodes=31,
    l2_regularization=0.01,
    early_stopping=False,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbose=0,
)
final_model.fit(X_full, y_full)
test_proba = final_model.predict_proba(X_test)

base_submission = pd.read_csv("data/submission_boosted_balanced.csv")
submission_rows = []

for _, row in selected.iterrows():
    multipliers = np.ones(len(class_names), dtype="float32")
    multipliers[fit_idx] = row["fit_mult"]
    multipliers[unhealthy_idx] = row["unhealthy_mult"]
    test_pred = (test_proba * multipliers).argmax(axis=1)
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_labels

    output_path = Path(f"data/submission_bacc_fine_{row['variant']}.csv")
    submission.to_csv(output_path, index=False)

    dist = submission[TARGET_COL].value_counts(normalize=True).mul(100).to_dict()
    changed = int((submission[TARGET_COL] != base_submission[TARGET_COL]).sum())
    submission_rows.append({
        "variant": row["variant"],
        "path": str(output_path),
        "fit_mult": row["fit_mult"],
        "unhealthy_mult": row["unhealthy_mult"],
        "val_balanced_accuracy": row["balanced_accuracy"],
        "changed_vs_boosted": changed,
        "changed_pct": round(changed / len(submission) * 100, 2),
        "test_at_risk_pct": dist.get("at-risk", 0),
        "test_fit_pct": dist.get("fit", 0),
        "test_unhealthy_pct": dist.get("unhealthy", 0),
    })

submission_report = pd.DataFrame(submission_rows).sort_values("val_balanced_accuracy", ascending=False)
submission_report

,variant,path,fit_mult,unhealthy_mult,val_balanced_accuracy,changed_vs_boosted,changed_pct,test_at_risk_pct,test_fit_pct,test_unhealthy_pct
0,fine_best,data/submission_bacc_fine_fine_best.csv,1.17,1.03,0.910935,3589,1.21,74.396878,11.544431,14.058691
5,unhealthy_one_step_down,data/submission_bacc_fine_unhealthy_one_step_down.csv,1.15,1.04,0.910829,3461,1.17,74.440158,11.438937,14.120905
3,fit_one_step_down,data/submission_bacc_fine_fit_one_step_down.csv,1.14,1.05,0.910781,3486,1.18,74.431705,11.380104,14.188191
1,public_anchor,data/submission_bacc_fine_public_anchor.csv,1.15,1.05,0.910733,3660,1.24,74.372872,11.438937,14.188191
4,unhealthy_one_step_up,data/submission_bacc_fine_unhealthy_one_step_up.csv,1.15,1.06,0.910704,3835,1.30,74.313701,11.438937,14.247362
7,less_fit_more_unhealthy,data/submission_bacc_fine_less_fit_more_unhealthy.csv,1.13,1.07,0.910697,3711,1.25,74.355628,11.327696,14.316676
2,fit_one_step_up,data/submission_bacc_fine_fit_one_step_up.csv,1.16,1.05,0.910695,3806,1.29,74.323506,11.488303,14.188191
6,both_slight_up,data/submission_bacc_fine_both_slight_up.csv,1.16,1.06,0.910666,3981,1.35,74.264335,11.488303,14.247362
